# Incompressible LES: Pitz Daily with WALE

## Preamble

In [1]:
# Standard Library
from pathlib import Path
import foamnordic as fno
import hedgehogs as hdg

### Directories

In [2]:
# Case Type and Name
CASE_TYPE = "les"
CASE_NAME = "pitzDailyWALE"
CATEGORY = "incompressible"

# FoamNordic Project Directory
PROJECT_DIR = Path.cwd().resolve()
BASE_DIR = next((path for path in (PROJECT_DIR, *PROJECT_DIR.parents) if (path / "tutorials" / "openfoam_tutorials").is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the FoamNordic source tree")
MAIN_DIR = BASE_DIR / "tutorials" / "foamnordic_tutorials" / CATEGORY
OF_SCRIPT_DIR = BASE_DIR / "tutorials" / "openfoam_tutorials" / CATEGORY / CASE_TYPE / CASE_NAME

# Output Directory
MODEL_DIR = MAIN_DIR / "model"
OUTPUT_DIR = MAIN_DIR / "output"

for directory in [MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

### Configuration

In [3]:
# Hedgehogs configuration
hdg.set_style()

In [4]:
# HPC configuration
ACCOUNT = "<allocation-account>"  # e.g. project_xxxxxxx
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "2G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

model_scheduler = fno.Slurm.model(
    cpus_per_task=1,
    mem_per_cpu=MEM_PER_CPU
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=model_scheduler
)

## Example - Closure Modelling

### WALE Closure

In [ ]:
# WALE model coefficient
C_W = 0.325

In [5]:
# WALE function for subgrid-scale turbulence modelling
def wale_function(velocity_grad, delta, C_w=C_W):
    gradient_squared = fno.Math.matmul(left=velocity_grad, right=velocity_grad)
    traceless_squared = fno.Math.dev(value=fno.Math.symm(value=gradient_squared))
    strain = fno.Math.symm(value=velocity_grad)

    sd2 = fno.Math.maximum(left=fno.Math.ddot(left=traceless_squared, right=traceless_squared), right=0.0)
    s2 = fno.Math.maximum(left=fno.Math.ddot(left=strain, right=strain), right=0.0)

    numerator = sd2**1.5
    denominator = s2**2.5 + sd2**1.25
    safe_denominator = fno.Math.where(condition=denominator > 0.0, when_true=denominator, when_false=1.0)

    nut = (C_w * delta)**2 * numerator / safe_denominator

    return fno.Math.where(condition=denominator > 0.0, when_true=nut, when_false=0.0)

In [6]:
# Define the WALE closure
wale_closure = fno.Closure(
    name="nutFjord",
    operator=fno.Operator.function(function=wale_function),
    inputs={
        "velocity_grad": fno.Field.grad(value="U"),
        "delta": fno.Field.delta(),
    },
    outputs={
        "nut": fno.Field(name="nut"),
    },
    key=None,
)

### Case Definition

In [7]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="pimpleFoam",
    ranks=N_TASKS,
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [8]:
# Connect the OpenFOAM case and execution resources
longship = fno.Longship(
    case=case,
    closures=(wale_closure,),
    scheduler=scheduler,
    verbose=True,
)

Resource,OpenFOAM,Model,Allocation
Processes,1 tasks,1 host(s),1 node(s)
CPU,1 cores,1 cores,2 cores
Memory,2 GiB,2 GiB,4.03125 GiB
Field programs,-,1,1
SHM,-,~32 MiB,~32 MiB


In [9]:
# Launch the workload
run = longship.launch(verbose=None)

[FoamNordic] nutFjord execution lane: native OpenFOAM.
[FoamNordic] Preparing mesh with blockMesh: pitzDailyWALE
[FoamNordic] Mesh is ready: pitzDailyWALE
[FoamNordic] Sailing submitted with Job ID: 1289860
[FoamNordic] Sailing has launched with Job ID: 1289860
[FoamNordic] Sailing started at: 2026-09-13T14:37:16
[FoamNordic] Sailing in background: pitzDailyWALE


In [10]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [11]:
# Summary of the job result
result.summary(style="compact", display=True);

Job ID,Name,Status,Partition,Node,Elapsed
1289860,pitzDailyWALE,succeeded,small,rc5130,00:01:10


### Post-processing

In [12]:
# Postprocessing
post = result.postprocess

velocity = post.field(name="U", time_idx=-1, physical_time=None)
pressure = post.field(name="p", time_idx=-1, physical_time=None)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    fields=["U", "p", "nut"],
    time_idx=-1,
    physical_time=None,
    verbose=True,
)

U shape: (12225, 3)
p shape: (12225,)


Field,Min,Max,Mean,Std,RMS
U,1.164906e-02,1.346690e+01,6.395988e+00,2.619908e+00,6.911771e+00
p,-3.771660e+01,1.129640e+02,6.460432e+01,2.186080e+01,6.820273e+01
nut,6.524120e-20,1.382990e-04,2.205728e-06,1.084035e-05,1.106248e-05
